In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import time
from datetime import datetime
import os

# Sklearn imports
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# TOP 3 Algoritmos identificados
from sklearn.svm import SVC
from sklearn.linear_model import RidgeClassifier, LogisticRegression

print("🚀 HYPERPARAMETER TUNING - TOP 3 ALGORITMOS")
print("=" * 60)

# Carregar dados
train_path = "/Users/marcelosilva/Desktop/projectOne/6/B- Binary Target DS DT/DatasetTrain.csv"
df_train = pd.read_csv(train_path)

print(f"✅ Dados carregados: {df_train.shape}")

# Target
target_col = 'status_nutricional_who'
print(f"✅ Target distribution: {df_train[target_col].value_counts().to_dict()}")

# Limpar dados
exclude_cols = ['id_anon', 'vd_zimc']
df_clean = df_train.drop(columns=exclude_cols, errors='ignore')

# TOP Feature Sets (baseado nos melhores resultados até agora)
feature_sets = {
    'top_3': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso'],
    'top_5': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso', 'k07_peso_final', 'imc_final_gravidico'],
    'top_8': ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso', 'k07_peso_final', 
              'imc_final_gravidico', 'peso_concordancia', 'j03_cor', 'indice_ponderal']
}

print(f"✅ Feature sets para tuning: {list(feature_sets.keys())}")

# Função de preprocessamento
def preprocess_data(df, features, target):
    """Preprocessa os dados para um conjunto específico de features"""
    
    # Verificar features disponíveis
    available_features = [f for f in features if f in df.columns]
    if len(available_features) != len(features):
        missing = set(features) - set(available_features)
        print(f"⚠️  Features não encontradas: {missing}")
    
    # Dataset com features + target
    df_subset = df[available_features + [target]].copy()
    
    # Separar features numéricas e categóricas
    numeric_features = df_subset.select_dtypes(include=[np.number]).columns.tolist()
    if target in numeric_features:
        numeric_features.remove(target)
    
    categorical_features = df_subset.select_dtypes(include=['object']).columns.tolist()
    
    # Tratar valores faltantes
    if len(numeric_features) > 0:
        imputer_num = SimpleImputer(strategy='median')
        df_subset[numeric_features] = imputer_num.fit_transform(df_subset[numeric_features])
    
    # Encoding de variáveis categóricas
    for col in categorical_features:
        le = LabelEncoder()
        df_subset[col] = df_subset[col].fillna('Missing')
        df_subset[col] = le.fit_transform(df_subset[col])
    
    X = df_subset.drop(columns=[target])
    y = df_subset[target]
    
    return X, y

# Configurar Grid Search
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'f1', 'roc_auc', 'precision', 'recall']

# Definir algoritmos e seus parâmetros para tuning
algorithms_tuning = {
    'SVM_RBF': {
        'model': SVC(kernel='rbf', probability=True, random_state=42),
        'params': {
            'C': [0.1, 1, 10, 100, 1000],
            'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}]
        }
    },
    
    'Ridge': {
        'model': RidgeClassifier(random_state=42),
        'params': {
            'alpha': [0.01, 0.1, 1, 10, 100, 1000],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}],
            'solver': ['auto', 'saga', 'lsqr']
        }
    },
    
    'LogisticRegression': {
        'model': LogisticRegression(random_state=42, max_iter=2000),
        'params': {
            'C': [0.01, 0.1, 1, 10, 100, 1000],
            'penalty': ['l1', 'l2', 'elasticnet'],
            'solver': ['liblinear', 'saga'],
            'class_weight': ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}],
            'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]  # Para elasticnet
        }
    }
}

print(f"✅ Algoritmos para tuning: {list(algorithms_tuning.keys())}")

# Resultados do tuning
tuning_results = []

# Executar Grid Search para cada combinação
total_combinations = len(feature_sets) * len(algorithms_tuning)
combination_count = 0

print(f"\n🧪 INICIANDO TUNING - {total_combinations} COMBINAÇÕES")
print("=" * 60)

start_time = time.time()

for feature_set_name, features in feature_sets.items():
    print(f"\n📊 Tuning para feature set: {feature_set_name} ({len(features)} features)")
    
    # Preprocessar dados
    try:
        X, y = preprocess_data(df_clean, features, target_col)
        print(f"   Dataset shape: {X.shape}")
        
        # Escalar features (importante para SVM)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        for alg_name, alg_config in algorithms_tuning.items():
            combination_count += 1
            
            print(f"\n🔧 {combination_count}/{total_combinations} | Tuning {alg_name} + {feature_set_name}")
            
            try:
                start_alg_time = time.time()
                
                # Configurar Grid Search
                model = alg_config['model']
                param_grid = alg_config['params']
                
                # Ajustar parâmetros específicos
                if alg_name == 'LogisticRegression':
                    # Filtrar combinações inválidas
                    valid_params = []
                    for penalty in param_grid['penalty']:
                        for solver in param_grid['solver']:
                            if penalty == 'elasticnet' and solver != 'saga':
                                continue
                            if penalty == 'l1' and solver not in ['liblinear', 'saga']:
                                continue
                            
                            base_params = {
                                'C': param_grid['C'],
                                'penalty': [penalty],
                                'solver': [solver],
                                'class_weight': param_grid['class_weight']
                            }
                            
                            if penalty == 'elasticnet':
                                base_params['l1_ratio'] = param_grid['l1_ratio']
                            
                            valid_params.append(base_params)
                    
                    # Grid Search com múltiplos param_grids
                    best_score = -1
                    best_params = None
                    best_model = None
                    
                    for params in valid_params:
                        try:
                            grid_search = GridSearchCV(
                                model, params, cv=cv_strategy, 
                                scoring='f1', n_jobs=-1, verbose=0
                            )
                            grid_search.fit(X_scaled, y)
                            
                            if grid_search.best_score_ > best_score:
                                best_score = grid_search.best_score_
                                best_params = grid_search.best_params_
                                best_model = grid_search.best_estimator_
                        except:
                            continue
                    
                    if best_model is None:
                        raise Exception("Nenhuma combinação válida encontrada")
                        
                else:
                    # Grid Search normal para SVM e Ridge
                    grid_search = GridSearchCV(
                        model, param_grid, cv=cv_strategy, 
                        scoring='f1', n_jobs=-1, verbose=0
                    )
                    grid_search.fit(X_scaled, y)
                    best_score = grid_search.best_score_
                    best_params = grid_search.best_params_
                    best_model = grid_search.best_estimator_
                
                end_alg_time = time.time()
                tuning_time = end_alg_time - start_alg_time
                
                # Avaliar melhor modelo com múltiplas métricas
                from sklearn.model_selection import cross_validate
                cv_results = cross_validate(
                    best_model, X_scaled, y, cv=cv_strategy,
                    scoring=['accuracy', 'f1', 'precision', 'recall', 'roc_auc'],
                    return_train_score=False,
                    n_jobs=-1  # ← PARALELIZAÇÃO ADICIONADA!
                )
                
                # Salvar resultados
                result = {
                    'combination_id': combination_count,
                    'feature_set': feature_set_name,
                    'n_features': len(features),
                    'algorithm': alg_name,
                    'best_f1_score': best_score,
                    'best_params': str(best_params),
                    'tuning_time': tuning_time,
                    'features_used': ', '.join(features),
                    
                    # Métricas detalhadas
                    'cv_accuracy_mean': cv_results['test_accuracy'].mean(),
                    'cv_accuracy_std': cv_results['test_accuracy'].std(),
                    'cv_f1_mean': cv_results['test_f1'].mean(),
                    'cv_f1_std': cv_results['test_f1'].std(),
                    'cv_precision_mean': cv_results['test_precision'].mean(),
                    'cv_precision_std': cv_results['test_precision'].std(),
                    'cv_recall_mean': cv_results['test_recall'].mean(),
                    'cv_recall_std': cv_results['test_recall'].std(),
                    'cv_auc_mean': cv_results['test_roc_auc'].mean(),
                    'cv_auc_std': cv_results['test_roc_auc'].std()
                }
                
                tuning_results.append(result)
                
                print(f"   ✅ F1: {best_score:.4f} | AUC: {cv_results['test_roc_auc'].mean():.4f} | Tempo: {tuning_time:.1f}s")
                print(f"   🎯 Melhores params: {best_params}")
                
            except Exception as e:
                print(f"   ❌ ERRO: {str(e)[:80]}")
                
                # Salvar erro
                tuning_results.append({
                    'combination_id': combination_count,
                    'feature_set': feature_set_name,
                    'n_features': len(features),
                    'algorithm': alg_name,
                    'best_f1_score': 0,
                    'error': str(e)[:200],
                    'tuning_time': 0,
                    'features_used': ', '.join(features)
                })
    
    except Exception as e:
        print(f"❌ Erro no preprocessamento para {feature_set_name}: {str(e)}")

# Análise dos resultados de tuning
total_time = time.time() - start_time

print(f"\n⏱️  TUNING CONCLUÍDO em {total_time/60:.1f} minutos")
print("=" * 60)

# Criar DataFrame com resultados
tuning_df = pd.DataFrame(tuning_results)

# Filtrar sucessos
success_df = tuning_df[tuning_df['best_f1_score'] > 0].copy()

if len(success_df) > 0:
    # Ordenar por F1-Score
    success_df = success_df.sort_values('cv_f1_mean', ascending=False)
    
    print(f"\n📈 RESULTADOS DO TUNING:")
    print(f"Total de combinações testadas: {len(tuning_df)}")
    print(f"Combinações bem-sucedidas: {len(success_df)}")
    
    # TOP 5 COMBINAÇÕES
    print(f"\n🏆 TOP 5 MELHORES COMBINAÇÕES (após tuning):")
    top_5 = success_df.head(5)
    
    for i, (_, row) in enumerate(top_5.iterrows(), 1):
        print(f"{i}. {row['algorithm']:15s} + {row['feature_set']:8s}")
        print(f"   F1: {row['cv_f1_mean']:.4f} (±{row['cv_f1_std']:.3f})")
        print(f"   Acc: {row['cv_accuracy_mean']:.4f} | AUC: {row['cv_auc_mean']:.4f}")
        print(f"   Params: {row['best_params']}")
        print()
    
    # Comparação com baseline
    print(f"\n📊 COMPARAÇÃO COM RESULTADOS ORIGINAIS:")
    print(f"🔥 Melhor F1 após tuning: {success_df['cv_f1_mean'].max():.4f}")
    print(f"🔥 Melhor AUC após tuning: {success_df['cv_auc_mean'].max():.4f}")
    print(f"🔥 Melhor Accuracy após tuning: {success_df['cv_accuracy_mean'].max():.4f}")
    print(f"📈 Baseline (classe majoritária): 75.2%")
    
    # Salvar resultados
    output_dir = "/Users/marcelosilva/Desktop/projectOne/6/E-Sklearn"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Salvar resultados do tuning
    tuning_df.to_csv(f"{output_dir}/hyperparameter_tuning_results_{timestamp}.csv", index=False)
    success_df.to_csv(f"{output_dir}/hyperparameter_tuning_best_{timestamp}.csv", index=False)
    
    # Salvar apenas o TOP 1 para uso final
    best_model_info = success_df.iloc[0]
    
    print(f"\n💾 ARQUIVOS SALVOS:")
    print(f"✅ hyperparameter_tuning_results_{timestamp}.csv - Todos os resultados")
    print(f"✅ hyperparameter_tuning_best_{timestamp}.csv - Apenas sucessos")
    
    print(f"\n🎯 MODELO CAMPEÃO:")
    print(f"🏆 {best_model_info['algorithm']} + {best_model_info['feature_set']}")
    print(f"🔥 F1-Score: {best_model_info['cv_f1_mean']:.4f}")
    print(f"🔥 AUC: {best_model_info['cv_auc_mean']:.4f}")
    print(f"🔥 Accuracy: {best_model_info['cv_accuracy_mean']:.4f}")
    
    print(f"\n🚀 PRÓXIMOS PASSOS:")
    print(f"1. Treinar modelo final com estes parâmetros")
    print(f"2. Avaliar no dataset de teste (ainda intocado)")
    print(f"3. Análise de feature importance")
    print(f"4. Curvas de aprendizado")

else:
    print(f"❌ Nenhuma combinação bem-sucedida!")

print(f"\n🏁 HYPERPARAMETER TUNING CONCLUÍDO!")

🚀 HYPERPARAMETER TUNING - TOP 3 ALGORITMOS
✅ Dados carregados: (3429, 39)
✅ Target distribution: {0: 2580, 1: 849}
✅ Feature sets para tuning: ['top_3', 'top_5', 'top_8']
✅ Algoritmos para tuning: ['SVM_RBF', 'Ridge', 'LogisticRegression']

🧪 INICIANDO TUNING - 9 COMBINAÇÕES

📊 Tuning para feature set: top_3 (3 features)
   Dataset shape: (3429, 3)

🔧 1/9 | Tuning SVM_RBF + top_3
   ✅ F1: 0.4146 | AUC: 0.6137 | Tempo: 324.2s
   🎯 Melhores params: {'C': 1, 'class_weight': {0: 1, 1: 4}, 'gamma': 0.01}

🔧 2/9 | Tuning Ridge + top_3
   ✅ F1: 0.4184 | AUC: 0.6189 | Tempo: 0.3s
   🎯 Melhores params: {'alpha': 0.01, 'class_weight': {0: 1, 1: 4}, 'solver': 'auto'}

🔧 3/9 | Tuning LogisticRegression + top_3
   ✅ F1: 0.4182 | AUC: 0.6176 | Tempo: 1.6s
   🎯 Melhores params: {'C': 0.01, 'class_weight': {0: 1, 1: 4}, 'penalty': 'l1', 'solver': 'liblinear'}

📊 Tuning para feature set: top_5 (5 features)
   Dataset shape: (3429, 5)

🔧 4/9 | Tuning SVM_RBF + top_5
   ✅ F1: 0.4132 | AUC: 0.5979 | Tempo

/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/pycar

   ✅ F1: 0.4161 | AUC: 0.6173 | Tempo: 1.3s
   🎯 Melhores params: {'alpha': 100, 'class_weight': {0: 1, 1: 4}, 'solver': 'auto'}

🔧 6/9 | Tuning LogisticRegression + top_5
   ✅ F1: 0.4171 | AUC: 0.6178 | Tempo: 18.2s
   🎯 Melhores params: {'C': 0.01, 'class_weight': {0: 1, 1: 4}, 'penalty': 'l2', 'solver': 'saga'}

📊 Tuning para feature set: top_8 (8 features)
   Dataset shape: (3429, 8)

🔧 7/9 | Tuning SVM_RBF + top_8
   ✅ F1: 0.4140 | AUC: 0.6004 | Tempo: 276.0s
   🎯 Melhores params: {'C': 0.1, 'class_weight': {0: 1, 1: 4}, 'gamma': 0.1}

🔧 8/9 | Tuning Ridge + top_8


/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/envs/pycar

   ✅ F1: 0.4155 | AUC: 0.6195 | Tempo: 1.5s
   🎯 Melhores params: {'alpha': 1000, 'class_weight': {0: 1, 1: 4}, 'solver': 'auto'}

🔧 9/9 | Tuning LogisticRegression + top_8
   ✅ F1: 0.4168 | AUC: 0.6192 | Tempo: 25.2s
   🎯 Melhores params: {'C': 0.01, 'class_weight': {0: 1, 1: 4}, 'penalty': 'l2', 'solver': 'liblinear'}

⏱️  TUNING CONCLUÍDO em 16.3 minutos

📈 RESULTADOS DO TUNING:
Total de combinações testadas: 9
Combinações bem-sucedidas: 9

🏆 TOP 5 MELHORES COMBINAÇÕES (após tuning):
1. Ridge           + top_3   
   F1: 0.4184 (±0.009)
   Acc: 0.4412 | AUC: 0.6189
   Params: {'alpha': 0.01, 'class_weight': {0: 1, 1: 4}, 'solver': 'auto'}

2. LogisticRegression + top_3   
   F1: 0.4182 (±0.011)
   Acc: 0.4686 | AUC: 0.6176
   Params: {'C': 0.01, 'class_weight': {0: 1, 1: 4}, 'penalty': 'l1', 'solver': 'liblinear'}

3. LogisticRegression + top_5   
   F1: 0.4171 (±0.005)
   Acc: 0.4482 | AUC: 0.6178
   Params: {'C': 0.01, 'class_weight': {0: 1, 1: 4}, 'penalty': 'l2', 'solver': 'saga'